<a href="https://colab.research.google.com/github/ojaspaul123/DL-journey/blob/main/ANN/Hyperparameter_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [59]:
import pandas as pd
import numpy as np

In [60]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("uciml/pima-indians-diabetes-database")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'pima-indians-diabetes-database' dataset.
Path to dataset files: /kaggle/input/pima-indians-diabetes-database


In [61]:
from google.colab import files
uploaded = files.upload()
df = pd.read_csv('diabetes.csv')
df.head()

Saving diabetes.csv to diabetes (1).csv


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [62]:
df.shape

(768, 9)

In [63]:
df.corr()['Outcome']

,Outcome
Pregnancies,0.221898
Glucose,0.466581
BloodPressure,0.065068
SkinThickness,0.074752
Insulin,0.130548
BMI,0.292695
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


In [64]:
X= df.iloc[:,:-1].values
y= df.iloc[:,-1].values

In [65]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X = sc.fit_transform(X)


In [66]:
X.shape

(768, 8)

In [67]:
from sklearn.model_selection import train_test_split

In [68]:
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size = 0.2, random_state = 42)

In [69]:
import tensorflow
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

**Basic Model**

In [70]:
model = Sequential()
model.add(Dense(32, activation='relu', input_dim=8))
model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [71]:
model.compile(optimizer='Adam', loss='binary_crossentropy', metrics=['accuracy'])

In [72]:
model.fit(X_train, y_train, batch_size=32, epochs=100,validation_data=(X_test, y_test))

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.6433 - loss: 0.6406 - val_accuracy: 0.6753 - val_loss: 0.6087
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6987 - loss: 0.5748 - val_accuracy: 0.6753 - val_loss: 0.5693
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7182 - loss: 0.5352 - val_accuracy: 0.6948 - val_loss: 0.5428
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7313 - loss: 0.5108 - val_accuracy: 0.7208 - val_loss: 0.5226
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7427 - loss: 0.4937 - val_accuracy: 0.7338 - val_loss: 0.5122
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7573 - loss: 0.4809 - val_accuracy: 0.7597 - val_loss: 0.5050
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7671 - loss: 0.4704 - val_accuracy: 0.7597 - val_loss: 0.4999
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7785 - loss: 0.4621 - val_accuracy: 0.7727 - 

In [73]:
pip install -U keras-tuner

In [74]:
import kerastuner as kt

**Select Optimizer**

In [75]:
def build_model(hp):
  model=Sequential()
  model.add(Dense(32,activation='relu',input_dim=8))

  model.add(Dense(1,activation='sigmoid'))
  op = hp.Choice('optimizer',values=['adam','sgd','rmsprop','adadelta'])
  model.compile(optimizer=op,loss='binary_crossentropy',metrics=['accuracy'])
  return model

In [76]:
tuner = kt.RandomSearch(build_model,objective='val_accuracy',max_trials=5)

Reloading Tuner from ./untitled_project/tuner0.json


In [77]:
tuner.search(X_train,y_train,epochs=10,validation_data=(X_test,y_test))

In [78]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'rmsprop'}

In [79]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [80]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [81]:
model.fit(X_train,y_train,epochs=100,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.7801 - loss: 0.4707 - val_accuracy: 0.7857 - val_loss: 0.5099
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7850 - loss: 0.4626 - val_accuracy: 0.7922 - val_loss: 0.5072
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7834 - loss: 0.4570 - val_accuracy: 0.7792 - val_loss: 0.5061
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7850 - loss: 0.4524 - val_accuracy: 0.7662 - val_loss: 0.5068
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7818 - loss: 0.4492 - val_accuracy: 0.7662 - val_loss: 0.5070
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7818 - loss: 0.4456 - val_accuracy: 0.7662 - val_loss: 0.5071
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7834 - loss: 0.4436 - val_accuracy: 0.7727 - val_loss: 0.5083
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.7834 - loss: 0.4416 - val_accuracy: 0.7

**No. of Nodes in a layers**

In [82]:
def build_model(hp):
  model=Sequential()
  units = hp.Int('units',min_value=8,max_value=128,step=8)
  model.add(Dense(units=units,activation='relu',input_dim=8))

  model.add(Dense(1,activation='sigmoid'))
  op = hp.Choice('optimizer',values=['adam','sgd','rmsprop','adadelta'])
  model.compile(optimizer='rmsprop',loss='binary_crossentropy',metrics=['accuracy'])
  return model

In [83]:
tuner = kt.RandomSearch(build_model,objective='val_accuracy',max_trials=5,directory='mydir')

Reloading Tuner from mydir/untitled_project/tuner0.json


In [84]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

In [85]:
tuner.get_best_hyperparameters()[0].values

{'units': 120, 'optimizer': 'sgd'}

In [86]:
model = tuner.get_best_models(num_models=1)[0]

In [87]:
model.fit(X_train,y_train,epochs=100,initial_epoch=6,validation_data=(X_test,y_test)  )

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.7622 - loss: 0.4594 - val_accuracy: 0.7727 - val_loss: 0.4971
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7704 - loss: 0.4504 - val_accuracy: 0.7403 - val_loss: 0.5002
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7687 - loss: 0.4465 - val_accuracy: 0.7468 - val_loss: 0.5024
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7785 - loss: 0.4437 - val_accuracy: 0.7597 - val_loss: 0.5011
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7736 - loss: 0.4412 - val_accuracy: 0.7597 - val_loss: 0.5017
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7801 - loss: 0.4379 - val_accuracy: 0.7662 - val_loss: 0.5017
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7785 - loss: 0.4361 - val_accuracy: 0.7662 - val_loss: 0.5038
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7818 - loss: 0.4338 - val_accuracy: 0.76

**How to select no. of layers**

In [88]:
def build_modell(hp):
  model = Sequential()

  model.add(Dense(72,activation='relu',input_dim=8))

  for i in range(hp.Int('num_layers',min_value=1,max_value=10)):
    model.add(Dense(72,activation='relu'))

  model.add(Dense(1,activation='sigmoid'))

  model.compile(optimizer='rmsprop',loss='binary_crossentropy',metrics=['accuracy'])
  return model

In [89]:
tuner_1 = kt.RandomSearch(
    build_modell,
    objective='val_accuracy',
    max_trials=5,
    directory='mydir',
    project_name='num_layers_search'
)

Reloading Tuner from mydir/num_layers_search/tuner0.json


In [90]:
tuner_1.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

In [91]:
tuner_1.get_best_hyperparameters()[0].values

{'num_layers': 6}

In [92]:
model = tuner_1.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [93]:
model.fit(X_train,y_train,epochs=100,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.7590 - loss: 0.4827 - val_accuracy: 0.7078 - val_loss: 0.5275
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7850 - loss: 0.4518 - val_accuracy: 0.7143 - val_loss: 0.5542
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7964 - loss: 0.4318 - val_accuracy: 0.7532 - val_loss: 0.5690
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7980 - loss: 0.4270 - val_accuracy: 0.7597 - val_loss: 0.5491
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7980 - loss: 0.4114 - val_accuracy: 0.7208 - val_loss: 0.5485
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8160 - loss: 0.4166 - val_accuracy: 0.7532 - val_loss: 0.5520
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8225 - loss: 0.3946 - val_accuracy: 0.7532 - val_loss: 0.5498
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8160 - loss: 0.3925 - val_accuracy: 0.75

In [101]:
def build_model(hp):
  model = Sequential()

  counter =0

  for i in range (hp.Int('num_layers',min_value=1,max_value=10)):
    if counter ==0:
      model.add(Dense(hp.Int('units'+str(i),min_value=8,max_value=128,step=8),
                      activation=hp.Choice('activation'+str(i),values=['relu','tanh','sigmoid']),
                      input_dim =8))
    else:
        model.add(Dense(hp.Int('units'+str(i),min_value=8,max_value=128,step=8),
                      activation=hp.Choice('activation'+str(i),values=['relu','tanh','sigmoid'])
              ))

        counter +=1


  model.add(Dense(1,activation='sigmoid'))

  model.compile(optimizer=hp.Choice('optimizer',values=['adam','sgd','rmsprop','adadelta']),
                loss='binary_crossentropy',metrics=['accuracy'])
  return model




In [102]:
tuner = kt.RandomSearch(build_model,objective='val_accuracy',max_trials=3,directory='mydir',project_name="final")

Reloading Tuner from mydir/final/tuner0.json


In [103]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

In [104]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 7,
 'units0': 24,
 'activation0': 'relu',
 'optimizer': 'rmsprop',
 'units1': 8,
 'activation1': 'relu',
 'units2': 8,
 'activation2': 'relu',
 'units3': 8,
 'activation3': 'relu',
 'units4': 8,
 'activation4': 'relu',
 'units5': 8,
 'activation5': 'relu',
 'units6': 8,
 'activation6': 'relu'}

In [105]:
model = tuner.get_best_models(num_models=1)[0]

In [106]:
model.fit(X_train,y_train,epochs=100,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.7410 - loss: 0.5556 - val_accuracy: 0.7662 - val_loss: 0.5377
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7524 - loss: 0.5214 - val_accuracy: 0.7468 - val_loss: 0.5209
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7606 - loss: 0.5017 - val_accuracy: 0.7597 - val_loss: 0.5107
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7736 - loss: 0.4876 - val_accuracy: 0.7597 - val_loss: 0.5031
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7883 - loss: 0.4793 - val_accuracy: 0.7597 - val_loss: 0.5003
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7850 - loss: 0.4714 - val_accuracy: 0.7532 - val_loss: 0.5022
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7785 - loss: 0.4668 - val_accuracy: 0.7727 - val_loss: 0.4982
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7834 - loss: 0.4616 - val_accuracy: 0.77